# Week 1 · Notebook 00 — Python for Astronomers, Crash Course
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/00_python_crash_course.ipynb)

**Who this is for:** anyone who has never written Python, or wrote some once and forgot.
**Time:** 3–4 hours, at your own pace. Seniors: skim, then help someone next to you.

By the end you can: store numbers in variables and arrays, write a function, select data with a
*boolean mask*, make a labelled plot, and — the payoff — build a fake planet transit from scratch
and estimate whether it is detectable.

> **How to use a notebook.** Click a grey code cell, press **Shift+Enter** to run it. Run cells
> top to bottom. If something breaks, *Runtime → Restart and run all* (Colab) fixes most mysteries.

In [ ]:
# --- Setup: run this cell first. Works in Google Colab and locally. ---
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR fork
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})
print("ready, repo root:", ROOT)

## 1. Variables, numbers, and units
Python is a calculator that remembers. A *variable* is a name for a value.

In [ ]:
R_sun_km = 695_700          # radius of the Sun in km (underscores are just for readability)
R_earth_km = 6_371
R_jupiter_km = 69_911

ratio = R_earth_km / R_sun_km
print("Earth/Sun radius ratio:", ratio)
print(f"Same thing, formatted: {ratio:.5f}")   # f-strings: put variables inside {}

**Physics moment.** When a planet passes in front of its star it blocks a fraction of the
star's disk. The fractional dip in brightness — the **transit depth** — is the ratio of areas:

$$\delta = \left(\frac{R_p}{R_\star}\right)^2$$

Areas go as radius *squared*, so small planets make *very* small dips.

In [ ]:
depth_earth = (R_earth_km / R_sun_km) ** 2      # ** means "to the power of"
depth_jup = (R_jupiter_km / R_sun_km) ** 2
print(f"Earth transiting the Sun:   {depth_earth*1e6:.0f} ppm  (parts per million)")
print(f"Jupiter transiting the Sun: {depth_jup*100:.2f} %")

### ✏️ Your turn 1
A "super-Earth" with radius 1.6 R⊕ orbits a small M-dwarf star with radius 0.3 R☉.
What is the transit depth in ppm? Why are M-dwarfs popular targets for finding small planets?

In [ ]:
# YOUR CODE HERE
R_planet_km = 1.6 * R_earth_km
R_star_km = 0.3 * R_sun_km
depth = None   # replace None with the formula
print(depth)

## 2. Lists, loops, and functions
A *function* packages a calculation so you can reuse it. Write the physics once, call it many times.

In [ ]:
def transit_depth(rp_rearth, rstar_rsun):
    """Fractional transit depth for a planet (Earth radii) and star (solar radii)."""
    rp_km = rp_rearth * R_earth_km
    rs_km = rstar_rsun * R_sun_km
    return (rp_km / rs_km) ** 2

for name, rp in [("Mars-size", 0.53), ("Earth", 1.0), ("Neptune", 3.9), ("Jupiter", 11.2)]:
    print(f"{name:10s} {transit_depth(rp, 1.0)*1e6:9.0f} ppm")

## 3. NumPy: math on whole arrays at once
A light curve is thousands of numbers. NumPy lets you do math on all of them in one line, fast,
without writing a loop. This is the single most important tool in the whole project.

In [ ]:
import numpy as np
time = np.linspace(0, 10, 11)          # 11 evenly spaced numbers from 0 to 10
print(time)
print(time * 2)                        # every element doubled
print(np.sin(time).round(2))           # every element through sin()
print("mean:", time.mean(), " max:", time.max(), " length:", len(time))

## 4. Boolean masks — the trick you will use every single week
Compare an array to a number and you get an array of `True`/`False`. Use it to *select* data.
Masks are how we pick in-transit points, clip outliers, and hide known planets.

In [ ]:
flux = np.array([1.00, 1.01, 0.99, 0.95, 0.94, 0.95, 1.00, 1.02])
low = flux < 0.97                  # a boolean mask
print(low)
print("the low points:", flux[low])
print("how many:", low.sum())       # True counts as 1
print("everything else:", flux[~low])  # ~ means NOT

## 5. Plotting with Matplotlib
Rule of the club: **every plot gets axis labels with units.** A plot without labels is a doodle.

In [ ]:
t = np.linspace(0, 10, 500)
f = 1 + 0.01 * np.sin(2 * np.pi * t / 3.0)
plt.plot(t, f, ".", ms=2, label="a star that wiggles")
plt.xlabel("Time [days]")
plt.ylabel("Normalised flux")
plt.legend()
plt.show()

## 6. Mini-project: build a fake transit light curve from scratch
Real data comes next week. First, make data where **you know the answer**. This idea —
*test your method on fake data with a known answer* — is the whole point of Week 4.

In [ ]:
rng = np.random.default_rng(42)           # random numbers you can reproduce

cadence_min = 30                           # one measurement every 30 minutes (like TESS FFIs)
time = np.arange(0, 27, cadence_min / 1440)   # 27 days (one TESS sector); 1440 minutes per day
period, t0, duration = 3.5, 1.2, 0.12      # days
depth = transit_depth(2.5, 1.0)            # a 2.5 R_earth planet around a Sun-like star
noise_ppm = 400

# Which points are inside a transit? Distance (in days) from the nearest mid-transit:
dt = np.abs(((time - t0 + 0.5 * period) % period) - 0.5 * period)
in_transit = dt < duration / 2

flux = np.ones_like(time)
flux[in_transit] -= depth                   # the box-shaped dip
flux += rng.normal(0, noise_ppm * 1e-6, len(time))   # add Gaussian noise

plt.plot(time, flux, ".", ms=2, color="k")
plt.xlabel("Time [days]"); plt.ylabel("Flux")
plt.title(f"Can you see the {depth*1e6:.0f} ppm transits? (noise = {noise_ppm} ppm per point)")
plt.show()

Probably not by eye! Each point's noise (400 ppm) is bigger than the dip (~530 ppm is close).
But there are many in-transit points. Averaging *N* points shrinks noise by √N:

$$\text{SNR} \approx \frac{\delta}{\sigma}\sqrt{N_\text{in-transit}}$$

Folding the light curve on the right period stacks all transits on top of each other:

In [ ]:
phase_days = ((time - t0 + 0.5 * period) % period) - 0.5 * period
plt.plot(phase_days * 24, (flux - 1) * 1e6, ".", ms=3, alpha=0.5)
plt.xlabel("Hours from mid-transit"); plt.ylabel("Flux − 1 [ppm]")
plt.xlim(-12, 12); plt.title("Folded on the true period"); plt.show()

N = in_transit.sum()
snr = depth / (noise_ppm * 1e-6) * np.sqrt(N)
print(f"{N} in-transit points -> expected SNR ~ {snr:.1f}")

### ✏️ Your turn 2 (the important one)
1. Fold on a **wrong** period (try 3.4 or 7.0). What happens to the dip? (Hint: 7.0 is special.)
2. Change the planet to 1.0 R⊕. Is it still visible after folding? What is the SNR?
3. Most surveys need SNR ≳ 7 to claim a detection. Find (by trying values) the smallest planet
   you could detect in this fake data set. Write the number down — you will measure this
   properly, for your own pipeline, in Week 4.

### ✅ Checkpoint (commit this notebook to your repo)
- [ ] Notebook runs top to bottom without errors
- [ ] Your Turn 1 and 2 answered in a markdown cell, in your own words
- [ ] One plot you are proud of, with labels and units

In [ ]:
# YOUR CODE HERE — experiment freely